# Deploying multilingual-e5-small on Databricks

## Overview

This notebook guides you through deploying the **multilingual-e5-small** text embedding model on Databricks. It is a lightweight, multilingual embedding model from [Microsoft Research](https://arxiv.org/abs/2402.05672) that maps text to 384-dimensional dense vectors.

### What is multilingual-e5-small?
- **Purpose**: Generate text embeddings for semantic search, retrieval, classification, and clustering
- **Languages**: 100+ languages
- **Size**: ~470MB (~134M parameters)
- **Output**: 384-dimensional embeddings
- **Max sequence length**: 512 tokens

### Important: Input Prefixing
For best results, prefix your inputs:
- **Queries**: `"query: What is Databricks?"`
- **Passages**: `"passage: Databricks is a unified analytics platform..."`

This asymmetric prefixing is how the model was trained and significantly improves retrieval quality.

### Deployment Process
1. **Create Cluster** (any ML cluster — no GPU required for download)
2. **Install Dependencies** (transformers, torch, sentence-transformers)
3. **Create Storage** (Unity Catalog schema and volume)
4. **Download & Register Model** (from Hugging Face, registered in MLflow)
5. **Create Endpoint** (Serverless or Provisioned Throughput)

### Time Estimate
- Setup: 5-10 minutes
- Model download & registration: 5-10 minutes
- Total: ~25-35 minutes

## Step 1: Create Cluster

**Before running this notebook, attach it to an ML cluster.**

### Cluster Configuration

1. Navigate to **Compute** in the Databricks left sidebar
2. Click **Create Cluster**
3. Configure with these settings:

   - **Cluster Name**: `E5 Embedding Deploy` (or any name you prefer)
   - **Policy**: `Unrestricted`
   - **Databricks Runtime**: `15.4 LTS ML` or later (ML runtime required)
   - **Worker Type**: Standard ML instance (no GPU required)
   - **Workers**: 1 (single worker is sufficient for this small model)
   - **Terminate After**: 30 minutes of inactivity

4. **Important**: You must use an ML runtime. Standard runtimes do not include MLflow and other ML libraries.

### After Cluster Creation
1. Wait for cluster to start (typically 3-5 minutes)
2. Attach this notebook to the cluster
3. Proceed to Step 2 below

## Step 2: Install Dependencies

Installing the required Python libraries. This cell will restart the Python kernel after installation.

In [ ]:
%pip install transformers torch sentence-transformers
%restart_python

## Step 3: Configuration

**Update the variables below with your environment details.**

In [ ]:
# ============================================================================
# CONFIGURATION - UPDATE THESE VALUES FOR YOUR ENVIRONMENT
# ============================================================================

# Unity Catalog Configuration
CATALOG_NAME = "YOUR_CATALOG_NAME"  # e.g., "shared", "main", "ml"
SCHEMA_NAME = "YOUR_SCHEMA_NAME"    # e.g., "embeddings", "models"
VOLUME_NAME = "multilingual_e5"     # Storage volume name

# Create catalog, schema, and volume if they don't exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")

# Model Configuration (DO NOT CHANGE)
MODEL_NAME = "intfloat/multilingual-e5-small"

# ============================================================================
# DERIVED PATHS - DO NOT EDIT
# ============================================================================
import os

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"
MODEL_STORAGE_PATH = os.path.join(VOLUME_PATH, "models", "multilingual-e5-small")
REGISTERED_MODEL_NAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.multilingual-e5-small-embeddings"

print("Configuration Summary:")
print(f"  Catalog: {CATALOG_NAME}")
print(f"  Schema: {SCHEMA_NAME}")
print(f"  Volume Path: {VOLUME_PATH}")
print(f"  Model Storage: {MODEL_STORAGE_PATH}")
print(f"  Registered Model: {REGISTERED_MODEL_NAME}")
print(f"\nVerify the paths above are correct before proceeding.")

## Step 4: Import Libraries

In [ ]:
import os
import mlflow
import transformers
import sentence_transformers
from sentence_transformers import SentenceTransformer

print(f"Transformers version: {transformers.__version__}")
print(f"Sentence Transformers version: {sentence_transformers.__version__}")
print(f"MLflow version: {mlflow.__version__}")

## Step 5: Download and Register Model

This step will:
1. **Download the model** from Hugging Face (~470MB, takes a few minutes)
2. **Register the model** in MLflow with Unity Catalog

### Why `llm/v1/embeddings`?
multilingual-e5-small is a text embedding model. Registering it with `llm/v1/embeddings` enables the standard Databricks embeddings API format, compatible with the OpenAI embeddings SDK.

### Timing
- Download: 2-5 minutes
- Registration: 3-5 minutes

**Do not interrupt this cell once started.**

In [ ]:
print(f"Starting model download and registration...")
print(f"  Model: {MODEL_NAME}")
print(f"  Storage: {MODEL_STORAGE_PATH}")
print(f"  This will take 5-10 minutes.\n")

# Step 1: Download model
print("Step 1/2: Downloading model (~470MB)...")
model = SentenceTransformer(MODEL_NAME, cache_folder=MODEL_STORAGE_PATH)
print(f"  Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"  Max sequence length: {model.max_seq_length}\n")

# Step 2: Register model in MLflow
print("Step 2/2: Registering model in MLflow...")
mlflow.set_registry_uri("databricks-uc")

# Create input/output examples for model signature
input_example = ["query: What is Databricks?", "passage: Databricks is a unified analytics platform."]

with mlflow.start_run():
    mlflow.sentence_transformers.log_model(
        model=model,
        artifact_path="model",
        task="llm/v1/embeddings",
        registered_model_name=REGISTERED_MODEL_NAME,
        input_example=input_example,
    )

print(f"\nModel registered successfully!")
print(f"  Registered as: {REGISTERED_MODEL_NAME}")
print(f"  Task type: llm/v1/embeddings")
print(f"\nNext step: Create a serving endpoint (see instructions below)")

## Step 6: Create Serving Endpoint

The model is now registered in Unity Catalog. The final step is to create a serving endpoint.

### Create via Databricks UI

1. Navigate to **Serving** in the left sidebar
2. Click **Create serving endpoint**
3. Configure:
   - **Endpoint Name**: `multilingual-e5-small-embeddings` (or your preferred name)
   - **Served entities**: Click **Select served entity**, go to **My models - Unity Catalog**, and find your registered model
   - **Endpoint Type**: **Serverless** (recommended — the model is small enough)
   - **Version**: Select the latest version
4. Click **Create**
5. Wait for the endpoint to become **Ready** (~5-10 minutes)

### Why Serverless Works Here
Unlike large LLMs (e.g., 70B parameter models), multilingual-e5-small is only ~470MB. Serverless can handle this efficiently, and it auto-scales to zero when not in use, minimizing costs.

### Testing the Endpoint

#### Via Python API
```python
from openai import OpenAI
import os

DATABRICKS_TOKEN = os.environ.get('DATABRICKS_TOKEN')
# Or in a notebook: dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url="https://<your-workspace>.cloud.databricks.com/serving-endpoints"
)

response = client.embeddings.create(
    model="multilingual-e5-small-embeddings",
    input=[
        "query: What is Databricks?",
        "passage: Databricks is a unified analytics platform for data engineering and data science."
    ]
)

for i, embedding in enumerate(response.data):
    print(f"Input {i}: {len(embedding.embedding)} dimensions")
    print(f"  First 5 values: {embedding.embedding[:5]}")
```

#### Via cURL
```bash
curl -X POST "https://<your-workspace>.cloud.databricks.com/serving-endpoints/multilingual-e5-small-embeddings/invocations" \
  -H "Authorization: Bearer $DATABRICKS_TOKEN" \
  -H "Content-Type: application/json" \
  -d '{"input": ["query: What is Databricks?", "passage: Databricks is a unified analytics platform."]}'
```

#### Computing Similarity
```python
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

query_emb = response.data[0].embedding
passage_emb = response.data[1].embedding

similarity = cosine_similarity(query_emb, passage_emb)
print(f"Similarity: {similarity:.4f}")
```

## Step 7 (Optional): Test Locally Before Serving

You can verify the model works correctly before creating a serving endpoint.

In [ ]:
# Test the model locally
test_inputs = [
    "query: What is machine learning?",
    "passage: Machine learning is a subset of artificial intelligence that enables systems to learn from data.",
    "query: ¿Qué es el aprendizaje automático?",  # Spanish
    "query: 机器学习是什么？",  # Chinese
]

embeddings = model.encode(test_inputs)

print(f"Generated {len(embeddings)} embeddings")
print(f"Embedding dimensions: {embeddings.shape[1]}")
print()

# Compute similarity matrix
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Similarity scores:")
for i in range(len(test_inputs)):
    for j in range(i + 1, len(test_inputs)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"  [{i}] vs [{j}]: {sim:.4f}")
        print(f"    '{test_inputs[i][:50]}...'")
        print(f"    '{test_inputs[j][:50]}...'")

## Deployment Summary

In [ ]:
print("=" * 60)
print("MULTILINGUAL-E5-SMALL DEPLOYMENT SUMMARY")
print("=" * 60)
print(f"\nConfiguration:")
print(f"  Catalog: {CATALOG_NAME}")
print(f"  Schema: {SCHEMA_NAME}")
print(f"  Volume: {VOLUME_NAME}")
print(f"\nModel Information:")
print(f"  Model: {MODEL_NAME}")
print(f"  Registered Name: {REGISTERED_MODEL_NAME}")
print(f"  Task Type: llm/v1/embeddings")
print(f"  Size: ~470MB (~134M parameters)")
print(f"  Embedding Dimensions: 384")
print(f"  Max Sequence Length: 512 tokens")
print(f"  Languages: 100+")
print(f"\nStorage Location:")
print(f"  Volume Path: {VOLUME_PATH}")
print(f"  Model Files: {MODEL_STORAGE_PATH}")
print(f"\nInput Prefixing (for best results):")
print(f"  Queries:   'query: <your question>'")
print(f"  Passages:  'passage: <your document text>'")
print(f"\nNext Steps:")
print(f"  1. Navigate to 'Serving' in Databricks UI")
print(f"  2. Create Serverless endpoint (recommended)")
print(f"  3. Select model: {REGISTERED_MODEL_NAME}")
print(f"  4. Wait for endpoint to become Ready (~5-10 min)")
print(f"  5. Test with sample embeddings")
print("=" * 60)